In [1]:
import pandas as pd
import psycopg2

In [ ]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""


                        CREATE TABLE IF NOT EXISTS dw.dim_canal_venda
                        (
                            id_canal_venda serial primary key,
                            canal_venda varchar(50)
                            
                        );
                        CREATE TABLE IF NOT EXISTS dw.dim_cliente
                        (
                            cliente_id serial primary key
                           
                        );

                        CREATE TABLE IF NOT EXISTS dw.dim_localidade
                        (
                            id_localidade serial primary key,
                            cidade varchar(50),
                            estado varchar(5)
                            
                        );
                        CREATE TABLE IF NOT EXISTS dw.dim_pagamento
                        (
                            id_pagamento serial primary key,
                            forma_pagamento varchar(30)
                            
                        );

                        CREATE TABLE IF NOT EXISTS dw.dim_produto   
                        (
                            produto_id serial primary key,
                            produto varchar(100),
                            categoria varchar(50),
                            marca varchar(50),
                            fornecedor varchar(50),
                            estoque_atual int,
                            estoque_minimo int,
                            centro_distribuicao varchar(100)
                        );
                         CREATE TABLE dw.dim_tempo (
                            data DATE primary key,
                            dia INT,
                            mes INT,
                            ano INT,
                            nome_mes VARCHAR(15),
                            trimestre VARCHAR(30),
                            dia_semana VARCHAR(20)
                        );

               """)
conexao.commit()
cursor.close()
conexao.close()

In [ ]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""

                    insert into dw.dim_produto (produto_id,produto, categoria, marca, fornecedor, estoque_atual, estoque_minimo, centro_distribuicao)
                    select distinct 
                        e.produto_id,
                        s.produto,
                        s.categoria_produto,
                        s.marca_produto,
                        e.fornecedor,
                        e.estoque_atual,
                        e.estoque_minimo,
                        e.centro_distribuicao
                    from staging.stg_vendas s 
                    left join staging.stg_estoque e
                        on s.produto = e.produto
                    where not exists (
                        select 1
                        from dw.dim_produto dp 
                        where dp.produto_id = s.produto_id
                );
               """)
               

cursor.execute("""                 
                     insert into dw.dim_cliente (cliente_id)
                     select distinct
                            s.cliente_id
                        from staging.stg_vendas s
                        where not exists (
                            select 1
                            from dw.dim_cliente dc  
                            where dc.cliente_id = s.cliente_id
                            
               );
                """)

cursor.execute("""      
                        SET lc_time = 'pt_BR.UTF-8';

                            INSERT INTO dw.dim_tempo (
                                data, dia, mes, ano, nome_mes, trimestre, dia_semana
                            )
                            SELECT DISTINCT
                                data,

                                EXTRACT(DAY FROM data),            
                                EXTRACT(MONTH FROM data),         
                                EXTRACT(YEAR FROM data),          

                                CASE EXTRACT(MONTH FROM data)     
                                    WHEN 1 THEN 'Janeiro'
                                    WHEN 2 THEN 'Fevereiro'
                                    WHEN 3 THEN 'Março'
                                    WHEN 4 THEN 'Abril'
                                    WHEN 5 THEN 'Maio'
                                    WHEN 6 THEN 'Junho'
                                    WHEN 7 THEN 'Julho'
                                    WHEN 8 THEN 'Agosto'
                                    WHEN 9 THEN 'Setembro'
                                    WHEN 10 THEN 'Outubro'
                                    WHEN 11 THEN 'Novembro'
                                    WHEN 12 THEN 'Dezembro'
                                END,
               
                                CASE
                                    WHEN EXTRACT(QUARTER FROM data) = 1 THEN 'Primeiro Trimestre'
                                    WHEN EXTRACT(QUARTER FROM data) = 2 THEN 'Segundo Trimestre' 
                                    WHEN EXTRACT(QUARTER FROM data) = 3 THEN 'Terceiro Trimestre'   
                                    WHEN EXTRACT(QUARTER FROM data) = 4 THEN 'Quarto Trimestre'
                                END,        

                                CASE EXTRACT(DOW FROM data)        
                                    WHEN 0 THEN 'Domingo'
                                    WHEN 1 THEN 'Segunda-feira'
                                    WHEN 2 THEN 'Terça-feira'
                                    WHEN 3 THEN 'Quarta-feira'
                                    WHEN 4 THEN 'Quinta-feira'
                                    WHEN 5 THEN 'Sexta-feira'
                                    WHEN 6 THEN 'Sábado'
                                END

                            FROM (
                                SELECT data_pedido AS data FROM staging.stg_vendas
                                UNION  
                                SELECT data_devolucao AS data FROM staging.stg_devolucoes
                            ) t
                            WHERE NOT EXISTS (
                                SELECT 1
                                FROM dw.dim_tempo dt
                                WHERE dt.data = t.data
                            );
               """)
             
cursor.execute("""          insert into dw.dim_localidade (cidade, estado)
                            select distinct
                               cidade,
                               estado
                            from staging.stg_vendas;
               """)
               
cursor.execute("""          insert into dw.dim_canal_venda (canal_venda)
                            select distinct
                                canal_vendas
                            from staging.stg_vendas;
               """)
               
cursor.execute("""          insert into dw.dim_pagamento (forma_pagamento)
                            select distinct
                                forma_pagamento
                            from staging.stg_vendas;
               



               """)
conexao.commit()
cursor.close()
conexao.close()